<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L06-eval-power-and-ranks/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L06-eval-power-and-ranks/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L06-eval-power-and-ranks/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L06-eval-power-and-ranks/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L06 · How many items does an eval need? Power, ranking uncertainty and a 100-item estimate of a 1,000-item score

**You will build:** the statistics a careful reader of a leaderboard needs: how many items it
takes to tell two models apart, how far a model's rank moves when you resample the items or
change the prompt format, and how a fitted item-response model lets a hundred well-chosen items
stand in for the whole benchmark.

**Time:** ~120 minutes · **Runs on:** a laptop CPU, no GPU, no download, far under 8 GiB ·
**Prerequisites:** `T07-L03` (the paired bootstrap) and `T01-L01` (power by simulation). Their
ideas come in as short given code, because a lesson cannot load another lesson's files.

**No language model runs here.** Every answer is simulated from an item-response model whose
parameters you can read, because a method for judging evaluations can only be checked in a world
where the truth is known, and on a real leaderboard nobody knows it. The lesson measures
methods; it says nothing about any real model.

By the end you will be able to:
1. Implement a two-parameter item-response model and the per-item variance of the measured gap
   between two models, with and without sharing the items.
2. Implement the number of items needed to detect a stated gap and a paired-bootstrap verdict,
   and measure by simulation the power they deliver.
3. Implement a bootstrap over items and prompt formats and a rank-frequency table, and measure
   how far each model's rank can move under each source of doubt.
4. Implement a posterior-mode ability estimate, a difficulty-stratified choice of 100 items and
   the performance-IRT estimator of a full-benchmark score, and measure their error against a
   random subset and against the noise no 100-item method can remove.
5. Explain, from your own printed table, why the plain accuracy on a curated subset is a
   biased estimate of the full score.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L03`, `T01-L01`

**This lesson assumes you can already:**

- write numpy array expressions with broadcasting, boolean masks and sums along an axis
- write the logistic (sigmoid) function and say what it does to the difference of two numbers
- say what a bootstrap resample is and read a percentile interval from the replicates (T07-L03 builds a paired bootstrap)
- read a statement such as a test with a 5% level and 80% power without deriving it (T01-L01 estimates power by simulation)
- take the derivative of a log-likelihood and apply one Newton step, new value = old value + slope / curvature (exercise 7 repeats it)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Adding Error Bars to Evals: A Statistical Approach to Language Model Evaluations](https://arxiv.org/abs/2411.00640) · Evan Miller (arXiv) · *free* — section 4.2 is the paired-differences result behind exercise 2, and section 5 and appendix B are the sample-size formula that exercise 3 implements
- [tinyBenchmarks: evaluating LLMs with fewer examples](https://arxiv.org/abs/2402.14992) · Polo et al. (arXiv) · *free* — section 3 on choosing evaluation examples and section 4 on the IRT model and the p-IRT estimator, which exercises 7 to 9 implement in one dimension
- [State of What Art? A Call for Multi-Prompt LLM Evaluation](https://arxiv.org/abs/2401.00595) · Mizrahi et al. (arXiv) · *free* — section 4 shows how model rankings change across semantically equivalent prompt templates, the second source of doubt that section 7 of this lesson resamples
- [The Basics of Item Response Theory, second edition](https://eric.ed.gov/?id=ED458219) · Frank B. Baker (ERIC; full text free) · *free* — chapter 2 on item characteristic curve models, chapter 5 on estimating an examinee's ability and chapter 6 on the information function
- [Item Response Theory: Foundations for Psychologists and Social Scientists, second edition](https://www.routledge.com/Item-Response-Theory-Foundations-for-Psychologists-and-Social-Scientists/Embretson-Reise/p/book/9780805853032) · Susan E. Embretson and Steven P. Reise (Routledge) · *paid* — chapter 4 on binary IRT models, chapter 6 on measuring persons and chapter 7 on estimating item parameters are the theory under p_correct, estimate_theta and the given fit_2pl

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed by the code itself.
import contextlib
import gc
import io
import math
import os
import signal
import sys
import threading
import time
from statistics import NormalDist
from typing import Callable

import numpy as np

print("python", sys.version.split()[0])
print("numpy ", np.__version__)

# Every check and every demo below runs through `_try`, so pressing Run all before you have
# written a line reaches the progress board at the foot of the notebook instead of stopping at
# the first stub.
# The nine exercises, in the order the notebook meets them: (label, function, where).
_EXERCISES = [("exercise 1", "p_correct", "section 2"),
              ("exercise 2", "diff_variances", "section 4"),
              ("exercise 3", "items_needed", "section 5"),
              ("exercise 4", "bootstrap_detects", "section 6"),
              ("exercise 5", "resample_scores", "section 7"),
              ("exercise 6", "rank_counts", "section 7"),
              ("exercise 7", "estimate_theta", "section 9"),
              ("exercise 8", "select_items", "section 9"),
              ("exercise 9", "p_irt_estimate", "section 9")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}
_STATUS: dict = {}       # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}   # label -> the stub whose NotImplementedError stopped it


def _unfinished(exc: BaseException) -> str:
    """The function that raised NotImplementedError: the innermost frame of the traceback.

    A demo calls several of your functions, so naming the stub that actually raised says which
    exercise to go back to.
    """
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (`items_needed`)"; several exercises -> "exercises 1, 2 and 3";
    a demo that has not run -> its own name."""
    if all(label in _FUNCTION for label in labels) and len(labels) > 1:
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} (`{_FUNCTION[label]}`)" if label in _FUNCTION else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so, by name. A wrong answer prints the
    check's own message, which names the likely mistake, and the notebook carries on, so one
    broken exercise never hides the feedback on the others. A demo names the exercises it
    `needs`: until each has passed its check, the demo says which one it is waiting for and
    skips, so it never runs on an answer a check has just rejected. Nothing is swallowed:
    every outcome is recorded for the progress board, and a script run exits non-zero there if
    any check came back wrong. A check still running after `_CHECK_SECONDS` (a demo:
    `_DEMO_SECONDS`) is stopped and counts as failed, so a loop that never ends cannot hang the
    notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        limit = _CHECK_SECONDS if label in _FUNCTION else _DEMO_SECONDS
        with _time_limit(limit, label):
            check()
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub]
        if owner and owner[0] != label:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises "
                  "NotImplementedError.")
        else:
            todo = f"`{stub}`" if stub else "a stub"
            print(f"{label}: not implemented yet, {todo} still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"
    if _STATUS[label] == "failed":
        gc.collect()        # a stopped runaway loop leaves what it appended in a reference cycle: free it now


def _demo(label: str, fn: Callable[[], None], needs: tuple = ()) -> None:
    """Run a demo cell through `_try` when this file is run as a notebook or a script, and skip it
    when the file is only imported (the autograder imports it to reach your functions). Given."""
    if __name__ == "__main__":
        _try(label, fn, needs)


# A check that never returns would freeze the notebook, and a loop that appends as it spins can
# eat gigabytes in seconds. Every exercise check runs under a time limit far above what a right
# answer needs (each takes milliseconds); a demo that simulates gets a longer one.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 30


class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _guarded(fn: Callable, seconds: float, what: str) -> Callable:
    """Wrap one of YOUR functions so that a single call that never returns fails, with a message,
    after `seconds`, instead of using up a whole demo's time limit. Given."""
    def run(*args, **kwargs):
        with _time_limit(seconds, what):
            return fn(*args, **kwargs)
    return run


def _refuse(*args, **kwargs):
    raise AssertionError("a given helper that already holds the answer (such as `_world_p`) is withdrawn "
                         "while this exercise is checked: write the formula in your own function")


@contextlib.contextmanager
def _withdrawn(*names: str):
    """While the block runs, the given helpers `names` (functions defined in this file) refuse to
    run: each one's code is swapped for `_refuse`, so calling it, aliasing it (`g = helper`) or
    wrapping it in another function all raise an AssertionError. The block receives the helpers
    themselves. An exercise that asks you to write a formula is checked this way, so that it cannot
    pass by calling a helper that already holds the formula. Given."""
    helpers = {name: globals()[name] for name in names}
    saved = {name: helper.__code__ for name, helper in helpers.items()}
    for helper in helpers.values():
        helper.__code__ = _refuse.__code__
    try:
        yield helpers                   # the withdrawn functions themselves, so a check can spot an alias
    finally:
        for name, helper in helpers.items():
            helper.__code__ = saved[name]

## 1. The benchmark we simulate

A benchmark is a list of items, and a model's score on it is the share it answers correctly.
This lesson gives every **item** two numbers, as the two-parameter logistic model of item
response theory does: a **difficulty** `b` (the ability at which a model answers it right half
the time) and a **discrimination** `a` (how sharply the chance of success climbs as ability
passes `b`). Every **model** gets one number, an **ability** `theta`. Then

`P(model answers item correctly) = sigmoid(a * (theta - b))`.

Given the item, a model answers once, by a coin flip with that probability. As of 14 October
2026 the tinyBenchmarks paper (arXiv:2402.14992) describes LLM benchmark responses with the
multidimensional form of this model, `sigmoid(alpha . theta - beta)`; the one-dimensional case
with `beta = a * b` is the model here (`claims.yaml`). In this lesson the numbers are planted,
so you can check every method against them.

The world has four parts, all generated by the cell below from one seed:

* a **universe** of 100,000 items from which any benchmark is a sample, and the **benchmark**
  the leaderboard uses, the first 1,000 of them;
* a **leaderboard** of twelve models with planted abilities, each answering every benchmark
  item under five **prompt formats**. Each model has its own **sensitivity**: the standard
  deviation of a shift in its effective ability, drawn once for each format. Nothing else changes
  between formats: a model's luck on an item is fixed, so a model with sensitivity zero would
  answer every item identically under all five;
* a **reference population** of 300 earlier models, whose answers we use to fit item
  parameters in section 8;
* 200 **held-out models**, used in section 9 to score the cheap estimators against the truth.

In [ ]:
SEED = 20261014          # this lesson's one seed (the authoring date): every item, every answer
                         # and every resample derives from it, so any machine prints the same
N_ITEMS = 1000           # the benchmark
N_UNIVERSE = 100_000     # the universe it is a sample of
N_FORMATS = 5            # prompt formats
N_BOOT = 400             # bootstrap replicates
ALPHA = 0.05             # two-sided level
TAIL = 10                # replicates a 95% interval leaves in each tail: 2.5% of N_BOOT
POWER = 0.80
K_SUBSET = 100           # items in the cheap subset
assert abs(TAIL - ALPHA / 2 * N_BOOT) < 1e-9


def _world_p(theta, a, b):
    """The world's own copy of the model, written with tanh so that the world exists before you
    write yours in exercise 1: sigmoid(x) = (1 + tanh(x / 2)) / 2. Given."""
    z = np.asarray(a, dtype=float) * (np.asarray(theta, dtype=float) - np.asarray(b, dtype=float))
    return 0.5 * (1.0 + np.tanh(0.5 * z))


def draw_items(rng, n):
    """Parameters of `n` items: discrimination `a` log-normal (median 1.8), difficulty `b` normal
    (mean -0.4, standard deviation 1.3). Given."""
    a = np.exp(rng.normal(np.log(1.8), 0.40, n))
    b = rng.normal(-0.4, 1.3, n)
    return a, b


UNIVERSE_A, UNIVERSE_B = draw_items(np.random.default_rng([SEED, 1]), N_UNIVERSE)
BANK_A, BANK_B = UNIVERSE_A[:N_ITEMS], UNIVERSE_B[:N_ITEMS]

MODEL_NAMES = ["Quill", "Ridge", "Sable", "Tarn", "Umber", "Vale",
               "Wren", "Yarrow", "Zephyr", "Argent", "Basalt", "Cinder"]
ABILITY = np.array([-1.30, -1.05, -0.60, -0.50, -0.20, -0.10, 0.15, 0.20, 0.55, 0.80, 0.85, 1.30])
SENSITIVITY = np.array([0.10, 0.30, 0.05, 0.25, 0.10, 0.35, 0.05, 0.30, 0.10, 0.20, 0.05, 0.15])
N_MODELS = len(MODEL_NAMES)


def make_leaderboard(rng):
    """Answers (0/1) of the twelve leaderboard models on the benchmark, one array of shape
    (formats, models, items). In each format a model's effective ability is its planted ability
    plus its sensitivity times a standard-normal draw. Each model gets ONE uniform draw per item,
    the same in every format, and answers the item correctly in a format when that draw falls
    below the probability the format's effective ability gives. So a model with sensitivity zero
    answers identically in every format, and any difference between formats comes from the
    format alone, not from a fresh coin flip. Also returns the effective abilities. Given."""
    effective = ABILITY[None, :] + SENSITIVITY[None, :] * rng.normal(size=(N_FORMATS, N_MODELS))
    p = _world_p(effective[:, :, None], BANK_A[None, None, :], BANK_B[None, None, :])
    luck = rng.random((N_MODELS, N_ITEMS))
    return (luck[None, :, :] < p).astype(np.int8), effective


def make_population(n_models, rng):
    """`n_models` models with abilities drawn from a standard normal, and their 0/1 answers on
    the benchmark, shape (models, items). Given."""
    theta = rng.normal(0.0, 1.0, n_models)
    p = _world_p(theta[:, None], BANK_A[None, :], BANK_B[None, :])
    return theta, (rng.random(p.shape) < p).astype(np.int8)


RESPONSES, EFFECTIVE = make_leaderboard(np.random.default_rng([SEED, 2]))
REF_THETA, REF_RESPONSES = make_population(300, np.random.default_rng([SEED, 3]))
TEST_THETA, TEST_RESPONSES = make_population(200, np.random.default_rng([SEED, 4]))


def show_world() -> None:
    print(f"universe: {N_UNIVERSE:,} items; benchmark: the first {N_ITEMS:,}; "
          f"{N_FORMATS} prompt formats; {N_MODELS} leaderboard models")
    print(f"discrimination a: median {np.median(UNIVERSE_A):.2f}, 5th to 95th percentile "
          f"{np.percentile(UNIVERSE_A, 5):.2f} to {np.percentile(UNIVERSE_A, 95):.2f}")
    print(f"difficulty b: mean {UNIVERSE_B.mean():.2f}, standard deviation {UNIVERSE_B.std():.2f}")
    print(f"accuracy at planted ability: over all {N_UNIVERSE:,} items; the last column: over the {N_ITEMS:,}-item "
          "benchmark\n")
    print(f"  {'model':<8} {'ability':>8} {'sensitivity':>12} {'accuracy at planted ability':>28} "
          f"{'accuracy on the benchmark, over the five formats (lowest to highest)':>50}")
    for m in range(N_MODELS):
        accuracy = _world_p(ABILITY[m], UNIVERSE_A, UNIVERSE_B).mean()
        per_format = RESPONSES[:, m, :].mean(axis=1)
        print(f"  {MODEL_NAMES[m]:<8} {ABILITY[m]:8.2f} {SENSITIVITY[m]:12.2f} {accuracy:28.3f} "
              f"{per_format.min():>35.3f} to {per_format.max():.3f}")


show_world()

Read the last column before going on, and compare it with the sensitivity column. The same
model, answering the same items with the same luck, scores differently under different formats,
and only its sensitivity can be the reason. A sensitivity is a standard deviation and each model
has only five shifts, so the realised spread is a noisy reading of it: read the column as a
tendency, not as a measurement. As of 14 October
2026 Sclar et al. (arXiv:2310.11324) report performance differences of up to 76 accuracy points
between formats for one real model, and that how well a format performs correlates only weakly
between models: a format that suits one model need not suit its neighbour, so changing the format
can reorder a leaderboard (`claims.yaml`). The planted sensitivities are a small-scale cousin of
that. Section 7 measures what they do to ranks.

## 2. Exercise 1 — `p_correct()`: the model, as a function

Every later section rests on one formula. Write it so that it works on numbers and on
arrays: ability `theta`, discrimination `a` and difficulty `b` may each be a single number or
an array, and numpy broadcasting should do the rest. For example `theta` of shape `(models, 1)`
against `a` and `b` of shape `(items,)` should give a `(models, items)` table of probabilities.

<details><summary>💡 Hint 1 — what to think about</summary>

The discrimination multiplies the *gap* between ability and difficulty, not the ability alone.
When ability equals difficulty the probability is one half, whatever the discrimination.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Form `a * (theta - b)`, then pass it through the logistic function, one over one plus the
exponential of its negative. Nothing needs a loop.
</details>

In [ ]:
def p_correct(theta, a, b):
    """Probability that a model of ability `theta` answers an item of discrimination `a` and
    difficulty `b` correctly, under the two-parameter logistic model: sigmoid(a * (theta - b)).

    All three arguments may be numbers or numpy arrays and are broadcast against each other.

    Example:
        >>> float(p_correct(0.0, 1.0, 0.0))
        0.5
        >>> round(float(p_correct(1.0, 2.0, 0.0)), 4)
        0.8808
        (ability 1, difficulty 0, discrimination 2: sigmoid(2))

    Returns:
        the probability (or an array of them), shape given by broadcasting.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement p_correct")


def _check_p_correct() -> None:
    with _withdrawn("_world_p") as given:      # the world's own copy of the formula is not available to your answer
        assert all(p_correct is not helper for helper in given.values()), (
            "p_correct is the given helper under another name: write the formula in your own function")
        assert abs(float(p_correct(0.3, 1.7, 0.3)) - 0.5) < 1e-12, (
            "ability equal to difficulty must give one half, whatever the discrimination: is `a` "
            "multiplying the difference (theta - b), and is the sign of the exponent right?")
        assert abs(float(p_correct(1.0, 2.0, 0.0)) - 0.8807970779778823) < 1e-9, (
            f"p_correct(1, 2, 0) should be sigmoid(2) = 0.8808, got {float(p_correct(1.0, 2.0, 0.0)):.4f}")
        got = float(p_correct(2.0, 3.0, 1.0))
        assert abs(got - 0.9525741268224334) < 1e-9, (
            f"ability 2, discrimination 3, difficulty 1 is sigmoid(3 * (2 - 1)) = 0.9526, got "
            f"{got:.4f}. The discrimination multiplies the whole gap (theta - b), not theta alone")
        got = float(p_correct(6.0, 3.0, 0.0))
        want = 1.0 / (1.0 + math.exp(-18.0))
        assert abs(got - want) < 1e-12, (
            f"ability 6, discrimination 3, difficulty 0 is sigmoid(18) = {want:.8f}, got {got:.8f}: do not clip or "
            "round the probabilities, later exercises need them close to 1")
        table = np.asarray(p_correct(np.array([[-1.0], [0.0], [1.0]]), np.array([1.0, 2.0, 0.5, 3.0]),
                                     np.array([0.0, 0.5, -0.5, 1.0])))
        assert table.shape == (3, 4), (
            f"abilities of shape (3, 1) against items of shape (4,) should give shape (3, 4), got "
            f"{table.shape}: let numpy broadcast, do not loop or reshape")
        assert np.all(np.diff(table, axis=0) > 0), "a more able model must be more likely to be right"
        assert np.all((table > 0) & (table < 1)), "probabilities lie strictly between 0 and 1"
        with np.errstate(all="ignore"):
            extreme = np.asarray(p_correct(np.array([40.0, -40.0]), 20.0, 0.0), dtype=float)
        assert np.all(np.isfinite(extreme)), (
            f"a very large |a * (theta - b)| gave {extreme}: exp(z) / (1 + exp(z)) overflows to nan where "
            "1 / (1 + exp(-z)) does not")
    print("exercise 1 looks right")


_try("exercise 1", _check_p_correct)

## 3. A score is a sample

One model's score on one benchmark is one draw. As of 14 October 2026 Miller
(arXiv:2411.00640) treats evaluation questions as drawn from an unseen super-population
(`claims.yaml`); the universe of this lesson is that population. Draw another benchmark of the
same size from it and the score moves. The cell below does that many times for one model and
compares the spread of the scores with the textbook standard error of a proportion,
`sqrt(p * (1 - p) / n)`. It uses the world's own function, not yours, so it runs whatever you
have written so far.

In [ ]:
INCUMBENT = 0.0          # the model the "challengers" of sections 4 to 6 are compared with


def accuracy_half_width(p, n, level=0.95):
    """Half-width of the usual interval (95% by default) for a proportion `p` measured on `n`
    independent items: the normal quantile times the standard error."""
    return NormalDist().inv_cdf(0.5 + level / 2.0) * math.sqrt(p * (1.0 - p) / n)


def show_a_score_is_a_sample() -> None:
    rng = np.random.default_rng([SEED, 10])
    p_universe = _world_p(INCUMBENT, UNIVERSE_A, UNIVERSE_B)
    true_accuracy = float(p_universe.mean())
    n_benchmarks = 2000
    idx = rng.integers(0, N_UNIVERSE, size=(n_benchmarks, N_ITEMS))
    scores = (rng.random(idx.shape) < p_universe[idx]).mean(axis=1)
    print(f"a model of ability {INCUMBENT}: accuracy over the whole universe {true_accuracy:.3f}")
    print(f"scores on {n_benchmarks} different {N_ITEMS}-item benchmarks drawn from it: lowest "
          f"{scores.min():.3f}, highest {scores.max():.3f}")
    print(f"spread (standard deviation) of those scores: {100 * scores.std():.2f} points; "
          f"textbook standard error: {100 * math.sqrt(true_accuracy * (1 - true_accuracy) / N_ITEMS):.2f} points")
    print(f"so one score is only good to about +/- {100 * accuracy_half_width(true_accuracy, N_ITEMS):.1f} "
          "points at 95%")


show_a_score_is_a_sample()

## 4. Exercise 2 — `diff_variances()`: why two models scored on the same items need fewer

Take models A and B. For item `i` let `x_A` be A's chance of getting it right, the number your
`p_correct` returns, and `x_B` B's. Two things make a measured score uncertain: which items the
benchmark happens to contain (the `x` vary from item to item) and the coin flip on each one.

* **Unpaired**: A and B are scored on separate random items. One model's score on one item
  then has variance `pbar * (1 - pbar)`, where `pbar` is the mean of its `x`, and the variance
  of the gap is the sum for A and for B.
* **Paired**: both models answer the SAME items, and the quantity measured is the per-item
  difference `d = (A's answer) - (B's answer)`. Given the item the two coin flips are
  independent, so `Var(d) = Var(x_A - x_B) + mean(x_A (1 - x_A)) + mean(x_B (1 - x_B))`, where
  the first term is the spread of `x_A - x_B` across items.

The two differ by exactly twice the covariance of `x_A` and `x_B` across items: what is hard
for A is hard for B, and the paired design cancels that shared part. As of 14 October 2026 this
is the result Miller (arXiv:2411.00640) gives for comparing two language models (`claims.yaml`).
The gap measured on `n` items has variance `(per-item variance) / n`.

<details><summary>💡 Hint 1 — what to think about</summary>

The unpaired variance uses the MEAN of each model's probabilities, then `pbar * (1 - pbar)`.
That is not the mean of `x * (1 - x)`: the second leaves out the variation between items.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For the paired variance take the difference of the two probability arrays, its variance across
items (divide by the number of items, not one fewer), and add the two within-item terms. Check
yourself on two identical models: the paired variance should be the within-item terms alone.
</details>

In [ ]:
def diff_variances(p_a, p_b):
    """Per-item variance of the measured accuracy gap between model A and model B.

    `p_a[i]` and `p_b[i]` are the two models' probabilities of answering item `i` correctly.
    Returns `(var_unpaired, var_paired)`:
      * unpaired: pbar_A (1 - pbar_A) + pbar_B (1 - pbar_B), with pbar the mean over items;
      * paired: the variance over items of (p_a - p_b), dividing by the number of items, plus
        the mean of p_a (1 - p_a) plus the mean of p_b (1 - p_b).

    Example:
        >>> diff_variances([1.0, 0.0], [1.0, 0.0])
        (0.5, 0.0)
        (both models solve the same item and miss the same item: scored on separate items each
        looks like a coin flip, scored on the same items they never differ)

    Returns:
        a pair of floats.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement diff_variances")


def _check_diff_variances() -> None:
    def exact(p_a, p_b):
        """The same two variances by brute force: list every outcome of (an item, A's answer, B's
        answer) with its probability, and take the variance of the gap's contribution. It lives
        inside this check, and is far too slow for a real benchmark."""
        n = len(p_a)
        mean = second = 0.0
        for i in range(n):
            for ans_a in (0, 1):
                for ans_b in (0, 1):
                    pr = (p_a[i] if ans_a else 1 - p_a[i]) * (p_b[i] if ans_b else 1 - p_b[i]) / n
                    mean += pr * (ans_a - ans_b)
                    second += pr * (ans_a - ans_b) ** 2
        va = float(np.mean(p_a)) * (1 - float(np.mean(p_a)))      # a Bernoulli with the mean of x
        vb = float(np.mean(p_b)) * (1 - float(np.mean(p_b)))
        return va + vb, second - mean ** 2

    # The unpaired variance first, then the paired one, so that each message names the mistake of
    # the quantity it tests.
    got = diff_variances([1.0, 0.0], [1.0, 0.0])[0]
    assert abs(got - 0.5) < 1e-12, (
        f"two identical models that solve one item and miss the other: the unpaired variance is 0.5, got {got}. "
        "Take the MEAN probability of each model first (0.5 here), then pbar * (1 - pbar) and add the two "
        "models; the mean of x * (1 - x) leaves out the variation between items")
    got = diff_variances([0.5, 0.5], [0.5, 0.5])[0]
    assert abs(got - 0.5) < 1e-12, (
        f"two models that flip a fair coin on every item: the unpaired variance is 0.25 + 0.25 = 0.5, got {got}: "
        "pbar * (1 - pbar) for model A PLUS the same for model B")
    got = diff_variances([1.0, 0.0], [1.0, 0.0])[1]
    assert abs(got - 0.0) < 1e-12, (
        f"two identical models that solve one item and miss the other: the paired variance is 0.0, got {got}. "
        "It is the spread across items of (p_a - p_b), which is zero here, plus the within-item terms "
        "mean(p_a (1 - p_a)) and mean(p_b (1 - p_b)), which are zero here too; it is not the unpaired "
        "pbar * (1 - pbar) terms, and a spread across items divides by n, not n - 1 (np.cov and "
        "np.var(..., ddof=1) divide by n - 1)")
    got = diff_variances([0.5, 0.5], [0.5, 0.5])[1]
    assert abs(got - 0.5) < 1e-12, (
        f"two models that flip a fair coin on every item: the paired variance is 0.5, got {got}. The "
        "paired variance keeps the within-item terms, mean(x (1 - x)), for BOTH models")
    got = diff_variances([1.0, 0.0], [0.0, 1.0])[1]
    assert abs(got - 1.0) < 1e-12, (
        f"models that disagree on every item: the paired variance is 1.0, got {got}. It is the variance across "
        "items of (p_a - p_b), dividing by the number of items n (np.var with ddof=0), not n - 1, "
        "plus the two within-item terms")
    rng = np.random.default_rng(3)
    p_a, p_b = rng.random(5), rng.random(5)
    want, have = exact(p_a, p_b), diff_variances(p_a, p_b)
    assert abs(have[0] - want[0]) < 1e-9, (
        f"unpaired variance {have[0]:.5f}, exact {want[0]:.5f}: take the MEAN probability of each "
        "model first, then pbar * (1 - pbar); the mean of x * (1 - x) leaves out the variation "
        "between items")
    assert abs(have[1] - want[1]) < 1e-9, (
        f"paired variance {have[1]:.5f}, exact {want[1]:.5f}: variance over items of (p_a - p_b), "
        "dividing by the number of items (not one fewer), plus mean(p_a (1 - p_a)) plus "
        "mean(p_b (1 - p_b))")
    print("exercise 2 looks right")


_try("exercise 2", _check_diff_variances)

Two models, one challenger and one incumbent, a few points apart. The cell below finds the
challenger's ability from the gap you ask for (given code), asks your `diff_variances` for the
two per-item variances, and then checks them against a simulation: it draws many benchmarks of
one size (the header of the table says how many) and measures the gap both ways, once with the
models sharing items and once with each model on its own. If the numbers agree, the formulas
are right and so is your code.

In [ ]:
def theta_for_gap(theta_base, gap):
    """Ability of a model whose accuracy over the universe is `gap` above that of ability
    `theta_base`, found by bisection and rounded to six places so that every machine finds the same
    value. Given."""
    target = float(_world_p(theta_base, UNIVERSE_A, UNIVERSE_B).mean()) + gap
    lo, hi = theta_base, theta_base + 8.0
    for _ in range(50):
        mid = 0.5 * (lo + hi)
        if float(_world_p(mid, UNIVERSE_A, UNIVERSE_B).mean()) < target:
            lo = mid
        else:
            hi = mid
    return round(0.5 * (lo + hi), 6)


def _universe_pair(gap):
    """Per-item probabilities over the universe of a challenger `gap` above the incumbent, and the
    challenger's ability. Uses YOUR p_correct."""
    theta_c = theta_for_gap(INCUMBENT, gap)
    return p_correct(theta_c, UNIVERSE_A, UNIVERSE_B), p_correct(INCUMBENT, UNIVERSE_A, UNIVERSE_B), theta_c


def show_pairing() -> None:
    p_a, p_b, theta_c = _universe_pair(0.05)
    var_unpaired, var_paired = diff_variances(p_a, p_b)
    n, sims = 400, 600
    rng = np.random.default_rng([SEED, 11])
    paired, unpaired = [], []
    for _ in range(sims):
        i = rng.integers(0, N_UNIVERSE, n)
        j = rng.integers(0, N_UNIVERSE, n)
        score_a = (rng.random(n) < p_a[i]).mean()
        score_b_same = (rng.random(n) < p_b[i]).mean()
        score_b_own = (rng.random(n) < p_b[j]).mean()
        paired.append(score_a - score_b_same)
        unpaired.append(score_a - score_b_own)
    da, db = p_a - p_a.mean(), p_b - p_b.mean()
    corr = float(np.mean(da * db) / math.sqrt(np.mean(da * da) * np.mean(db * db)))
    print(f"challenger ability {theta_c}, incumbent {INCUMBENT}; true gap over the universe: "
          f"{100 * (p_a.mean() - p_b.mean()):.2f} points")
    print(f"{sims} benchmarks of {n} items; the gap measured on each, in points")
    print(f"  {'design':<9} {'your variance per item':>23} {'predicted SD of the gap':>25} "
          f"{'simulated SD of the gap':>25}")
    for name, var, sim in (("unpaired", var_unpaired, unpaired), ("paired", var_paired, paired)):
        print(f"  {name:<9} {var:>23.4f} {100 * math.sqrt(var / n):>22.2f} pts "
              f"{100 * np.std(sim):>22.2f} pts")
    print(f"correlation of the two models' per-item probabilities over the universe: {corr:.3f}")
    print(f"paired variance as a share of the unpaired one: {var_paired / var_unpaired:.3f}")


_demo("pairing", show_pairing, needs=("exercise 1", "exercise 2"))

The simulated spread should sit close to the predicted one in both rows (the spread itself, from
that many benchmarks, is only known to a few per cent). Look at the correlation line: it is what the
paired design feeds on. These two models are independent once the item is fixed, so all of
their agreement comes from shared difficulty. Models that also share training data and
mistakes would agree more, which this lesson does not simulate.

## 5. Exercise 3 — `items_needed()`: how many items to see a stated gap

Suppose the true gap between challenger and incumbent is `delta` accuracy points, and the
per-item variance of the measured gap is `var`. A test at two-sided level `alpha` that you
want to catch the gap with probability `power` needs, to a good approximation,

`n = (z_(1 - alpha/2) + z_power)^2 * var / delta^2`,

where `z_q` is the point below which a share `q` of a standard normal lies. The first `z` is how
many standard errors from zero the measured gap must sit before you call it real; the second is
how many standard errors above that threshold the true gap must sit for a share `power` of
repeats to clear it. As of 14
October 2026 Miller (arXiv:2411.00640) gives this as his sample-size formula, with the
paired per-item variance in the numerator (`claims.yaml`). Round up: a fraction of an item is
a whole item.

<details><summary>💡 Hint 1 — what to think about</summary>

The standard library has the normal distribution: `statistics.NormalDist().inv_cdf` turns a
share into a `z`. The gap may be given with either sign, and a gap of zero cannot be detected
with any number of items: raise an error rather than divide by zero.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Two quantiles (one tail of the two-sided level, and the power), add them, square the sum,
multiply by the variance, divide by the square of the gap, and round up to a whole number.
Test with other values than the usual 5% and 80%: a typed 1.96 and 0.84 will not follow them.
</details>

In [ ]:
def items_needed(delta, var, alpha=0.05, power=0.8):
    """Items needed to detect a true accuracy gap `delta` with a two-sided test of level `alpha`
    and probability `power`, when the measured gap has per-item variance `var`.

    n = ceil((z(1 - alpha/2) + z(power))**2 * var / delta**2), z(q) the standard normal quantile.
    `delta` may be negative (only its size matters). Raises ValueError if `delta` is 0.

    Example:
        >>> items_needed(0.05, 0.23)
        723
        (the two quantiles are 1.960 and 0.842, and the formula gives 722.1: a partial item is a
        whole item, so the answer is rounded UP to 723, not to the nearest whole number)

    Returns:
        an int.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement items_needed")


def _check_items_needed() -> None:
    z = NormalDist().inv_cdf
    # Each assertion isolates one mistake, so the first message that fires names it. The first two are
    # about the shape of the formula. The next two compare with the UNROUNDED quotient to within one
    # item, so that only a wrong quantile can fail them, not the rounding; the round-up assertion
    # comes after them.
    base = items_needed(0.05, 0.24)
    assert items_needed(0.025, 0.24) > 3.9 * base, (
        "halving the gap should need about four times the items: delta is SQUARED in the denominator")
    assert items_needed(0.05, 0.48) > 1.9 * base, (
        "twice the per-item variance should need about twice the items: var multiplies, it is not under a root")
    quotient = (z(0.95) + z(0.5)) ** 2 * 0.24 / 0.05 ** 2
    got = items_needed(0.05, 0.24, alpha=0.10, power=0.5)
    assert abs(got - quotient) < 1, (
        f"level 0.10, power 0.5 (so the power quantile is zero), variance 0.24, gap 0.05: the formula gives "
        f"{quotient:.1f}, got {got}. Only the level's quantile is left: z(1 - alpha/2) = z(0.95) for a TWO-sided "
        "level, from NormalDist().inv_cdf (do not type 1.96); the sum of the two quantiles is squared")
    quotient = (z(0.975) + z(0.90)) ** 2 * 0.24 / 0.05 ** 2
    got = items_needed(0.05, 0.24, alpha=0.05, power=0.90)
    assert abs(got - quotient) < 1, (
        f"level 0.05, power 0.90, variance 0.24, gap 0.05: the formula gives {quotient:.1f}, got {got}. The "
        "power quantile is z(power) = z(0.90), from NormalDist().inv_cdf: not z(1 - power) and not a typed 0.84")
    assert items_needed(0.05, 0.23) == 723, (
        f"delta 0.05, variance 0.23, 5% and 80%: (1.960 + 0.842)^2 * 0.23 / 0.05^2 = 722.1, and a "
        f"partial item is a whole item: round UP to 723 (math.ceil), not to the nearest whole number or down "
        f"(722); got {items_needed(0.05, 0.23)}")
    assert items_needed(0.05, 0.25) == 785, (
        f"delta 0.05, variance 0.25, 5% and 80%: (1.960 + 0.842)^2 * 0.25 / 0.05^2 = 784.9, rounded "
        f"up to 785; got {items_needed(0.05, 0.25)}")
    assert items_needed(-0.05, 0.25) == 785, "only the size of the gap matters, not its sign"
    got = items_needed(0.05, 0.25)
    assert isinstance(got, (int, np.integer)) and not isinstance(got, bool), (
        f"return a whole number of items, an int; got {type(got).__name__}")
    try:
        items_needed(0.0, 0.25)
    except ValueError:
        pass
    else:
        raise AssertionError("a gap of zero cannot be detected: raise ValueError")
    print("exercise 3 looks right")


_try("exercise 3", _check_items_needed)

Now use it. The table asks for four stated gaps, builds a challenger for each (given code),
measures both per-item variances with your `diff_variances` and turns them into items with your
`items_needed`. Then it turns the formula around, with the given helper `min_detectable_gap`,
to ask the question a reader of a leaderboard has: with the benchmark we have, how small a gap
can be told from noise?

In [ ]:
def min_detectable_gap(n, var, alpha=ALPHA, power=POWER):
    """The smallest true gap that `n` items detect with probability `power`: the sample-size
    formula solved for `delta`. Given."""
    z = NormalDist().inv_cdf
    return (z(1.0 - alpha / 2.0) + z(power)) * math.sqrt(var / n)


def show_items_needed() -> None:
    print(f"incumbent ability {INCUMBENT}; level {ALPHA}, power {POWER}; items needed to detect "
          "the stated gap")
    print(f"  {'stated gap':>10} {'challenger':>11} {'unpaired':>9} {'paired':>8} "
          f"{'paired / unpaired':>18} {'paired, vs the first row':>26}")
    first = None
    for gap in (0.08, 0.06, 0.04, 0.02):
        p_a, p_b, theta_c = _universe_pair(gap)
        true_gap = float(p_a.mean() - p_b.mean())
        var_unpaired, var_paired = diff_variances(p_a, p_b)
        n_unpaired, n_paired = items_needed(true_gap, var_unpaired), items_needed(true_gap, var_paired)
        first = n_paired if first is None else first
        print(f"  {100 * true_gap:>8.1f} pts {theta_c:>11.3f} {n_unpaired:>9} {n_paired:>8} "
              f"{n_paired / n_unpaired:>18.3f} {n_paired / first:>25.2f}x")
    print(f"\nthe smallest gap {N_ITEMS} items can detect (same level and power), from the per-item variances "
          "of a 4-point gap above three different incumbents:")
    print(f"  {'incumbent accuracy':>19} {'unpaired design':>17} {'paired design':>15}")
    for theta_base in (INCUMBENT - 1.0, INCUMBENT, INCUMBENT + 1.0):
        p_a = p_correct(theta_for_gap(theta_base, 0.04), UNIVERSE_A, UNIVERSE_B)
        p_b = p_correct(theta_base, UNIVERSE_A, UNIVERSE_B)
        var_unpaired, var_paired = diff_variances(p_a, p_b)
        print(f"  {p_b.mean():>19.3f} {100 * min_detectable_gap(N_ITEMS, var_unpaired):>14.1f} pts "
              f"{100 * min_detectable_gap(N_ITEMS, var_paired):>12.1f} pts")


_demo("items needed", show_items_needed, needs=("exercise 1", "exercise 2", "exercise 3"))

Read the last column against the first column: when the gap halves, by what factor do the items
grow? Then read the last three lines against your own sense of what a leaderboard difference is.
A benchmark of the size we simulate cannot reliably tell apart (at this level and power) two
models closer than those lines say, whatever the leaderboard prints. The distance depends on
where on the scale the pair sits, because the per-item variance does: the three rows are three
different distances, not one distance for every pair. Compare each with the gaps between
neighbouring models of a similar accuracy in the table of section 1.

## 6. Exercise 4 — `bootstrap_detects()`: the verdict of a paired bootstrap

The formula is a plan. The test you will actually run is a paired bootstrap, which is the one
you met in `T07-L03`, over items instead of tasks: draw a list of item indices with replacement,
apply the SAME list to both models' 0/1 answers, take the difference in the number correct, and
repeat many times. `paired_bootstrap_sums` below does that and hands you one integer per
replicate: A's correct answers minus B's on that replicate's items.

Your function reads the verdict by counting. A is better when fewer than `tail` replicates lie
**on or below** zero, where `tail` is 2.5% of the replicates, one tail of a 95% interval (the
lesson sets it as a whole number); B is better when fewer than `tail` lie on or above it. A
replicate whose difference is exactly zero does not show a gap, so it counts against the claim.
Count integers, never percentiles: a percentile interval gives the same verdict except at the
boundary, where it interpolates between two replicates, and only the count is exact there.

<details><summary>💡 Hint 1 — what to think about</summary>

Two counts: how many replicates sum to zero or less, and how many to zero or more. A
replicate at exactly zero belongs to both counts.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

If the first count is below `tail`, A is better; else if the second is below `tail`, B is
better; otherwise there is no verdict. Compare with "fewer than", not "at most".
</details>

In [ ]:
def paired_bootstrap_sums(x_a, x_b, n_boot, rng):
    """T07-L03's paired bootstrap over items, on integers. Draws ONE grid of resampled item
    indices, `n_boot` rows of `len(x_a)` draws with replacement, and applies it to BOTH models'
    0/1 answers; each row gives A's correct answers minus B's on the resampled items. Using one
    grid for both is what keeps the models paired. Given."""
    n = len(x_a)
    idx = rng.integers(0, n, size=(n_boot, n))
    return x_a[idx].sum(axis=1) - x_b[idx].sum(axis=1)


def bootstrap_detects(boot_sums, tail):
    """The verdict of a paired bootstrap on whether model A is better than model B.

    `boot_sums` holds one integer per replicate: A's correct answers minus B's. `tail` is how many
    replicates a 95% interval leaves beyond zero on one side.
      * return +1 (A better) if fewer than `tail` replicates are <= 0;
      * otherwise return -1 (B better) if fewer than `tail` replicates are >= 0;
      * otherwise return 0 (no verdict).
    A replicate exactly at zero counts on BOTH sides. (`tail` is a small share of the replicates,
    so both conditions are never true together; the order above is just the order of the checks.)

    Example:
        >>> bootstrap_detects([5, 4, 6, 3, 0, 5, 7, 2, 4, 5], 2)
        1
        >>> bootstrap_detects([5, 4, 6, 3, 0, 5, 7, 2, 4, 5], 1)
        0
        (one replicate is at zero: it is fewer than 2 but not fewer than 1)

    Returns:
        +1, -1 or 0, as an int.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement bootstrap_detects")


def _check_bootstrap_detects() -> None:
    sums = [5, 4, 6, 3, 0, 5, 7, 2, 4, 5]
    assert bootstrap_detects(sums, 2) == 1, (
        "one replicate at zero and a tail of 2: fewer than 2 replicates are on or below zero, so "
        f"A is better, got {bootstrap_detects(sums, 2)!r}")
    assert bootstrap_detects(sums, 1) == 0, (
        "with a tail of 1 the single replicate at exactly zero is NOT fewer than 1: a replicate "
        "at zero counts against the claim, so compare with <= 0 and then with 'fewer than'")
    assert bootstrap_detects([-3, -1, -4, -2, -6, -5, -1, -2, -3, -7], 2) == -1, (
        "every replicate negative: B is better, return -1")
    assert bootstrap_detects([-2, 3, 4, -1, 2, 5, -3, 1, 2, 1], 2) == 0, (
        "replicates on both sides of zero: no verdict, return 0")
    mirrored = [-5, -4, -6, -3, 0, -5, -7, -2, -4, -5]
    assert bootstrap_detects(mirrored, 2) == -1 and bootstrap_detects(mirrored, 1) == 0, (
        "the mirror image of the first case: B is better when fewer than `tail` replicates are on or ABOVE zero, "
        "and a replicate at zero counts there too")
    assert bootstrap_detects(np.array([1, 2, 3, 4]), 1) == 1, "arrays are as good as lists"
    # a replicate count of exactly `tail` on the wrong side is not enough
    ten_at_or_below = [0] * 10 + [4] * 390
    nine_at_or_below = [0] * 9 + [4] * 391
    assert bootstrap_detects(ten_at_or_below, 10) == 0, (
        "10 of 400 replicates on or below zero with a tail of 10: not FEWER than 10, so no verdict")
    assert bootstrap_detects(nine_at_or_below, 10) == 1, (
        "9 of 400 replicates on or below zero with a tail of 10: fewer than 10, so A is better")
    print("exercise 4 looks right")


_try("exercise 4", _check_bootstrap_detects)

`simulate_power` (given) is the loop you would write after `T01-L01`, with your verdict inside
it: draw a benchmark of `n_items` items from the universe, let both models answer, run the paired
bootstrap, count how often it says "A is better". `normal_power` is the closed-form power of the
normal approximation, the number the sample-size formula is built to hit. The cell runs two
stated gaps at half, exactly and twice the number of items the formula asked for.

In [ ]:
def simulate_power(theta_a, theta_b, n_items, n_sims, rng):
    """Share of simulated `n_items`-item benchmarks on which YOUR bootstrap_detects says model A
    is better than B, and the share on which it says B is better. Given."""
    wins = reverse = 0
    for _ in range(n_sims):
        idx = rng.integers(0, N_UNIVERSE, n_items)
        a, b = UNIVERSE_A[idx], UNIVERSE_B[idx]
        x_a = (rng.random(n_items) < _world_p(theta_a, a, b)).astype(np.int8)
        x_b = (rng.random(n_items) < _world_p(theta_b, a, b)).astype(np.int8)
        verdict = bootstrap_detects(paired_bootstrap_sums(x_a, x_b, N_BOOT, rng), TAIL)
        wins += int(verdict == 1)
        reverse += int(verdict == -1)
    return wins / n_sims, reverse / n_sims


def normal_power(delta, var, n, alpha=ALPHA):
    """Power of the normal approximation: the chance that a gap of `delta` measured on `n` items
    with per-item variance `var` clears the two-sided threshold. Given."""
    normal = NormalDist()
    return normal.cdf(abs(delta) * math.sqrt(n / var) - normal.inv_cdf(1.0 - alpha / 2.0))


def show_power() -> None:
    n_sims = 300
    print(f"{n_sims} simulated benchmarks per row, paired bootstrap with {N_BOOT} replicates, "
          f"level {ALPHA}; the formula aims at power {POWER}")
    print(f"  {'stated gap':>10} {'items':>7} {'(of formula)':>13} {'formula power':>14} "
          f"{'simulated power':>16} {'+/- (1 SE)':>11} {'wrong sign':>11}")
    shortfalls = []                    # (formula power - simulated power) / SE, rows not pinned near 1
    for gap in (0.08, 0.05):
        p_a, p_b, theta_c = _universe_pair(gap)
        true_gap = float(p_a.mean() - p_b.mean())
        _, var_paired = diff_variances(p_a, p_b)
        planned = items_needed(true_gap, var_paired)
        for share in (0.5, 1.0, 2.0):
            n = int(round(planned * share))
            rng = np.random.default_rng([SEED, 12, int(round(100 * gap)), int(10 * share)])
            power, wrong = simulate_power(theta_c, INCUMBENT, n, n_sims, rng)
            se = math.sqrt(power * (1.0 - power) / n_sims)
            formula = normal_power(true_gap, var_paired, n)
            print(f"  {100 * true_gap:>8.1f} pts {n:>7} {f'{share:g} x':>13} "
                  f"{formula:>14.3f} {power:>16.3f} {se:>11.3f} {wrong:>11.3f}")
            if formula < 0.95 and se > 0:
                shortfalls.append((formula - power) / se)
    print(f"in the {len(shortfalls)} rows whose formula power is below 0.95, the simulated power is below the formula's "
          f"in {sum(v > 0 for v in shortfalls)}, by up to {max(shortfalls):.1f} standard errors")


_demo("power", show_power, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

Compare the middle row of each gap with the power the formula aimed for. The simulated figure is
itself an estimate, with the standard error beside it, so a difference of a standard error or two
is noise, not a finding. The line under the table counts the rows that fall below the formula
and says by how many standard errors at most: read it as a direction as well as a size. The
verdict you wrote is slightly stricter than the test the formula describes. The bootstrap's sums
are whole numbers, and a replicate exactly at zero counts against the claim; and "fewer than
`tail`" allows at most `tail - 1` replicates where an exact 2.5% tail would hold `tail`. Each
moves the threshold by a fraction of a standard error of the gap, and the first fraction
shrinks as the benchmark grows. (Estimating the variance from the sample, which the formula
takes as known, costs almost nothing at sizes like these.) Use the formula to choose a size and
the simulation to confirm it before you spend the budget. The last column should be small at
every size: a test that often announces the wrong winner is broken, not powerful.

## 7. Exercises 5 and 6 — how far can a rank move?

A leaderboard prints one score per model and sorts them. That order is one draw from two kinds
of doubt. **Items**: another sample of items from the same universe would score the models a
little differently. **Format**: another meaning-preserving prompt format would too, and, as
section 1 showed, by a different amount for each model. As of 14 October 2026 Mizrahi et al.
(arXiv:2401.00595) report that a single instruction template gives unreliable model rankings
for many of the tasks they study (`claims.yaml`).

### Exercise 5 — `resample_scores()`

A bootstrap that expresses both kinds of doubt. In every replicate: choose one format
(uniformly, or format 0 if formats are not being resampled), choose the items (with replacement,
or all of them once if items are not being resampled), and score EVERY model by its number
correct on those items in that format. One format and one item draw per replicate, shared by
all models, because a leaderboard scores everyone on the same questions.

<details><summary>💡 Hint 1 — what to think about</summary>

The answers are an array of shape (formats, models, items). One replicate needs one format
number and one list of item numbers, and then a sum over the item axis for all models at once.
Draw them only from the generator you are given.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over replicates. In each: pick the format (or format 0), pick the item indices (or all of
them in order), take that format's slice, select those items for every model at once, and sum
over the items. Store one row per replicate.
</details>

In [ ]:
def resample_scores(responses, n_boot, rng, resample_items=True, resample_formats=True):
    """Bootstrap replicates of every model's number-correct score.

    `responses` is an integer array of 0/1 answers of shape (formats, models, items). Each of the
    `n_boot` replicates, using only `rng`:
      * picks one format: uniformly at random if `resample_formats`, else format 0;
      * picks the items: `n_items` indices drawn uniformly WITH replacement if `resample_items`,
        else every item exactly once;
      * scores every model by its number of correct answers on those items in that format.
    The same format and the same items are used for all models. `responses` is not modified.

    Example:
        >>> r = np.ones((2, 3, 4), dtype=int)
        >>> resample_scores(r, 2, np.random.default_rng(0), resample_items=False).tolist()
        [[4, 4, 4], [4, 4, 4]]
        (every model answers every item correctly, in every format)

    Returns:
        an integer array of shape (n_boot, models).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement resample_scores")


def _check_resample_scores() -> None:
    rng = np.random.default_rng(1)
    r = rng.integers(0, 2, size=(3, 4, 30))
    before = r.copy()
    out = np.asarray(resample_scores(r, 12, np.random.default_rng(2)))
    assert out.shape == (12, 4), f"expected one row per replicate and one column per model: (12, 4), got {out.shape}"
    assert np.array_equal(r, before), "the answers must not be modified"
    flat = np.asarray(resample_scores(r, 5, np.random.default_rng(3), resample_items=False,
                                      resample_formats=False))
    assert all(np.array_equal(row, r[0].sum(axis=1)) for row in flat), (
        "with neither source resampled every replicate is just format 0 on all items: "
        f"{r[0].sum(axis=1).tolist()}, got {flat[0].tolist()}")
    fmt = np.asarray(resample_scores(r, 60, np.random.default_rng(4), resample_items=False))
    totals = [tuple(r[f].sum(axis=1)) for f in range(3)]
    assert all(tuple(row) in totals for row in fmt), (
        "with items fixed each replicate must be one whole format's totals: pick the format "
        "ONCE per replicate, for all models, not once per model")
    assert {totals.index(tuple(row)) for row in fmt} == {0, 1, 2}, (
        "over 60 replicates every format should come up: choose uniformly among all formats")
    same = np.tile(rng.integers(0, 2, size=(1, 1, 50)), (2, 5, 1))
    both = np.asarray(resample_scores(same, 25, np.random.default_rng(5), resample_formats=False))
    assert all(len(set(row)) == 1 for row in both), (
        "five models with identical answers must tie in every replicate: draw ONE set of item "
        "indices per replicate and use it for every model")
    assert len({tuple(row) for row in both}) > 5, "resampling items should change the scores"
    again = np.asarray(resample_scores(r, 12, np.random.default_rng(2)))
    assert np.array_equal(out, again), "the same generator seed must give the same replicates"
    other = np.asarray(resample_scores(r, 12, np.random.default_rng(99)))
    assert not np.array_equal(out, other), "a different generator seed must give different replicates: draw from `rng`"
    opposite = np.stack([np.stack([same[0, 0], 1 - same[0, 0]])] * 2)
    pair = np.asarray(resample_scores(opposite, 20, np.random.default_rng(6), resample_formats=False))
    assert np.all(pair.sum(axis=1) == 50), (
        "two models with opposite answers on every item must have scores adding to the number of items in every "
        "replicate: exactly as many draws as items, one set of draws shared by both models, one format")
    last = np.zeros((1, 1, 20), dtype=int)
    last[0, 0, 19] = 1
    seen = np.asarray(resample_scores(last, 80, np.random.default_rng(7))).ravel()
    assert seen.max() > 0, "the last item never came up in 80 replicates: check the upper bound of your random draw"
    print("exercise 5 looks right")


_try("exercise 5", _check_resample_scores)

### Exercise 6 — `rank_counts()`

Replicate scores become a rank table. A model's rank in a replicate is
`1 + (the number of models whose score is strictly higher)`, so tied models share a rank. The
table counts, for every model, how many replicates gave it rank 1, rank 2, and so on.

<details><summary>💡 Hint 1 — what to think about</summary>

A rank is not a position in a sort. Two models with the same score must get the same rank, and
the next model's rank skips the places the tie used up. Ask of each model only: how many others
are strictly better?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

In each replicate count, for every model, how many scores are strictly larger and add one to get
its rank, then count one more in that model's cell for that rank. Ranks start at 1 but array
positions start at 0.
</details>

In [ ]:
def rank_counts(scores):
    """How often each model takes each rank across replicates.

    `scores` has shape (replicates, models); higher is better. In a replicate a model's rank is
    1 + the number of models with a strictly higher score, so tied models share a rank.

    Example:
        >>> rank_counts([[3, 3, 1]]).tolist()
        [[1, 0, 0], [1, 0, 0], [0, 0, 1]]
        (the first two models tie for rank 1; the third has two models above it, so it is rank 3)

    Returns:
        an integer array of shape (models, models) whose entry [m, r - 1] is the number of
        replicates in which model m had rank r.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement rank_counts")


def _check_rank_counts() -> None:
    got = np.asarray(rank_counts([[3, 3, 1]]))
    assert got.tolist() == [[1, 0, 0], [1, 0, 0], [0, 0, 1]], (
        f"scores 3, 3, 1: the first two share rank 1 and the third is rank 3 (two models are "
        f"above it); got {got.tolist()}. Ties share a rank: rank = 1 + the number STRICTLY higher")
    got = np.asarray(rank_counts([[10, 20, 30], [30, 20, 10]]))
    assert got.tolist() == [[1, 0, 1], [0, 2, 0], [1, 0, 1]], (
        f"a higher score is a better (smaller) rank number; got {got.tolist()}")
    rng = np.random.default_rng(6)
    scores = rng.integers(0, 4, size=(50, 6))
    got = np.asarray(rank_counts(scores))
    assert got.shape == (6, 6) and (got.sum(axis=1) == 50).all(), (
        "each model must be counted once in every replicate: every row sums to the number of "
        "replicates")
    assert got[:, 0].sum() >= 50, "somebody holds rank 1 in every replicate"
    print("exercise 6 looks right")


_try("exercise 6", _check_rank_counts)

The next cell runs the bootstrap three ways: items only, formats only, and both. For each
model it reports the range of ranks that remains after the same number of best and worst
replicates are dropped at each end (given helper `rank_range`; the table's header says how
many), beside its rank on the leaderboard (format 0, all items) and its rank by planted
ability. Then, for each pair of neighbours on that leaderboard, it counts how often the
lower-ranked model scores strictly higher.

In [ ]:
def rank_range(counts, tail):
    """The central range of ranks per model, from a table of rank counts. Drops the `tail` best
    and the `tail` worst replicates of each model and returns the best and worst rank that
    remains, as an integer array of shape (models, 2). Given."""
    out = []
    for row in np.asarray(counts):
        cum = np.cumsum(row)
        total = int(cum[-1])
        out.append((int(np.argmax(cum >= tail + 1)) + 1, int(np.argmax(cum >= total - tail)) + 1))
    return np.array(out)


def _ranks_of(scores):
    """The rank of each entry of a vector of scores, 1 for the highest, tied scores sharing a rank,
    found with YOUR rank_counts on a single replicate. Given."""
    return np.argmax(np.asarray(rank_counts([scores])), axis=1) + 1


def show_ranks() -> None:
    sources = (("items", dict(resample_items=True, resample_formats=False)),
               ("formats", dict(resample_items=False, resample_formats=True)),
               ("both", dict(resample_items=True, resample_formats=True)))
    board = _ranks_of(RESPONSES[0].sum(axis=1))
    planted = _ranks_of(ABILITY)
    ranges, swaps, spans, format_scores = {}, {}, {}, None
    order = [int(m) for m in np.lexsort((np.arange(N_MODELS), board))]
    for k, (name, kwargs) in enumerate(sources):
        scores = resample_scores(RESPONSES, N_BOOT, np.random.default_rng([SEED, 20 + k]), **kwargs)
        format_scores = scores if name == "formats" else format_scores
        ranges[name] = rank_range(rank_counts(scores), TAIL)
        spans[name] = int((ranges[name][:, 1] - ranges[name][:, 0] + 1).sum())
        swaps[name] = [float(np.mean(scores[:, order[i + 1]] > scores[:, order[i]]))
                       for i in range(N_MODELS - 1)]
    print(f"{N_BOOT} replicates per source; rank range = best and worst rank left after dropping "
          f"the {TAIL} best and the {TAIL} worst replicates")
    print(f"  {'board':>5} {'model':<8} {'planted':>8} {'sensitivity':>12}  "
          f"{'items':>10}{'formats':>10}{'both':>10}")
    for m in order:
        cells = "".join(f"{f'{ranges[n][m][0]} to {ranges[n][m][1]}':>10}" for n, _ in sources)
        print(f"  {board[m]:>5} {MODEL_NAMES[m]:<8} {planted[m]:>8} {SENSITIVITY[m]:>12.2f}  {cells}")
    print("  (board = rank on the leaderboard: format 0, all items; planted = rank by planted ability)")
    print(f"\nsum over the models of how many ranks each range covers ({N_MODELS} would mean no model ever moves):")
    print("  " + ", ".join(f"{name} {spans[name]}" for name, _ in sources))
    drawn = [int(np.sum((format_scores == RESPONSES[f].sum(axis=1)).all(axis=1))) for f in range(N_FORMATS)]
    print(f"  formats only: the {N_FORMATS} formats came up {min(drawn)} to {max(drawn)} times each in {N_BOOT} replicates, so "
          f"dropping {TAIL} at each end "
          + ("removes none of them: that range is the best and worst rank over the formats."
             if min(drawn) > TAIL else "can remove a whole format."))
    print("\nshare of replicates in which a leaderboard neighbour overtakes the model above it")
    print(f"  {'pair (lower overtakes upper)':<30} {'items':>7} {'formats':>8} {'both':>7}")
    for i in range(N_MODELS - 1):
        upper, lower = order[i], order[i + 1]
        print(f"  {MODEL_NAMES[lower] + ' over ' + MODEL_NAMES[upper]:<30} "
              + "".join(f"{swaps[n][i]:>{w}.3f}" for n, w in (("items", 7), ("formats", 9), ("both", 8))))


_demo("ranks", show_ranks, needs=("exercise 5", "exercise 6"))

Each model's luck on an item is the same in every format, so the formats column is the formats'
doing alone. A rank moves when the model's own score moves with the format and when a
neighbour's does, so read a model's neighbours as well as its sensitivity.

Three questions for the table. Which source of doubt widened the ranges more, and by how much
does the sum at the foot change? Which models move most under the formats, and does that line up
with the planted sensitivity column? And compare the `board` column with the `planted` one: the
leaderboard itself is one draw, so what it shows is not necessarily the order of the abilities.
A leaderboard that publishes only the rank column hides every range in this table.

## 8. Fitting the items from earlier models

Section 9's question is the one a team asks when an evaluation is too expensive to run in full:
can a hundred items stand in for the thousand? A new model's answers on a hundred items say
little unless you know how hard and how discriminating each of those items is. You learn that
from models that have already been run on the whole benchmark: here, the 300 reference models.

`fit_2pl` below is given. It fits every model's ability and every item's two numbers together
(a joint maximum-a-posteriori fit) by alternating Newton steps, writing each item's log-odds as
`a * theta + c` with `c = -a * b`. Weak Gaussian priors keep an item that every model got right
(or wrong) from running off to infinity. Only `a * (theta - b)` matters, so the origin and the
unit of the abilities are arbitrary: after each sweep the abilities are rescaled to mean 0 and
standard deviation 1, which fixes them. The fitted
values wear a hat: `a_hat`, `b_hat`. Read the function, then look at how well it recovered the
planted values: the planted values are not an input to the fit, and a real benchmark never
offers them for comparison.

In [ ]:
def fit_2pl(responses, n_sweeps=40):
    """Joint MAP fit of the two-parameter logistic model to a (models, items) table of 0/1
    answers. Priors: ability N(0, 1); discrimination N(1, 1), kept at or above 0.05; the
    intercept c = -a * b is N(0, 3^2). Each sweep: one Newton step for every ability, one 2 x 2
    Newton step for every item's (a, c), then abilities rescaled to mean 0 and standard
    deviation 1. Returns (a_hat, b_hat, theta_hat). Given."""
    y = np.asarray(responses, dtype=float)
    n_models, n_items = y.shape
    theta, a, c = np.zeros(n_models), np.ones(n_items), np.zeros(n_items)
    lam_a, lam_c = 1.0, 1.0 / 3.0 ** 2
    for _ in range(n_sweeps):
        p = 1.0 / (1.0 + np.exp(-(a[None, :] * theta[:, None] + c[None, :])))
        grad = (a[None, :] * (y - p)).sum(axis=1) - theta
        hess = (a[None, :] ** 2 * p * (1.0 - p)).sum(axis=1) + 1.0
        theta = theta + grad / hess
        p = 1.0 / (1.0 + np.exp(-(a[None, :] * theta[:, None] + c[None, :])))
        w = p * (1.0 - p)
        g_a = (theta[:, None] * (y - p)).sum(axis=0) - lam_a * (a - 1.0)
        g_c = (y - p).sum(axis=0) - lam_c * c
        h_aa = (theta[:, None] ** 2 * w).sum(axis=0) + lam_a
        h_ac = (theta[:, None] * w).sum(axis=0)
        h_cc = w.sum(axis=0) + lam_c
        det = h_aa * h_cc - h_ac ** 2
        a = np.maximum(a + (h_cc * g_a - h_ac * g_c) / det, 0.05)
        c = c + (-h_ac * g_a + h_aa * g_c) / det
        mean, sd = theta.mean(), theta.std()
        theta = (theta - mean) / sd
        c = c + a * mean
        a = a * sd
    return a, -c / a, theta


def _corr(x, y):
    """Pearson correlation from means alone (no matrix product, so no BLAS)."""
    dx, dy = x - x.mean(), y - y.mean()
    return float(np.mean(dx * dy) / math.sqrt(np.mean(dx * dx) * np.mean(dy * dy)))


A_HAT, B_HAT, _ = fit_2pl(REF_RESPONSES)
print(f"fitted on {REF_RESPONSES.shape[0]} reference models x {REF_RESPONSES.shape[1]} items")
print(f"correlation of fitted and planted difficulty:     {_corr(B_HAT, BANK_B):.3f}")
print(f"correlation of fitted and planted discrimination: {_corr(A_HAT, BANK_A):.3f}")
print(f"fitted discrimination: smallest {A_HAT.min():.2f}, largest {A_HAT.max():.2f}; planted: smallest "
      f"{BANK_A.min():.2f}, largest {BANK_A.max():.2f}")
_same = np.where(REF_RESPONSES.min(axis=0) == REF_RESPONSES.max(axis=0))[0]
if len(_same):
    print(f"{len(_same)} items were answered the same way by every reference model; their fitted difficulties "
          f"lie between {B_HAT[_same].min():.1f} and {B_HAT[_same].max():.1f}, held there by the priors, not by data")

The fit is good and not exact. Discrimination is the harder of the two: it is a slope, and 300
models give each item's slope only so many data points. An item that every reference model
answered the same way is, as far as the data say, easier (or harder) than every reference model,
but not by how much; the priors place it, and items with identical answer columns get identical
fitted values, which is why exercise 8 below fixes its tie rules. Everything in section 9 uses the fitted values `A_HAT` and `B_HAT`, as a team
with real models would, never the planted ones.

## 9. Exercises 7 to 9 — a hundred items standing in for a thousand

The target is one model's **realised** score on the whole benchmark: the share of its 1,000
answers that were right. We run it on `K_SUBSET` items only. Three pieces turn those answers
into an estimate of the target:

* **Exercise 7**, `estimate_theta`: from the model's answers on the chosen items and their fitted
  `a`, `b`, find the ability that makes them most likely, with a standard-normal prior pulling it
  gently towards zero (the posterior mode). Use Newton's method. `fit_2pl` above takes one such
  step for every reference model at once with the prior fixed; you write it for one model, with
  the width of the prior as an argument, and repeat it until the step is tiny.
* **Exercise 8**, `select_items`: choose which items to run.
* **Exercise 9**, `p_irt_estimate`: combine the answers we have with the model's predicted chance
  on the items we skipped. As of 14 October 2026 the tinyBenchmarks paper calls this the p-IRT
  estimator and derives it as a conditional expectation (`claims.yaml`).

The same paper reports that 100 curated examples suffice to estimate an LLM's performance on
MMLU (`claims.yaml`); here you test the idea in a world where the answer is known. It compares
stratified random sampling, clustering of the examples and IRT-based estimators, and the
selection rule of exercise 8 is a simple one-dimensional cousin of the first and the last.

### Exercise 7 — `estimate_theta()`

<details><summary>💡 Hint 1 — what to think about</summary>

Each item pulls the ability up when the model got it right more often than the current ability
predicts, and down when less often; the pull is bigger for a more discriminating item. The
prior adds a pull back towards zero, and a curvature that keeps the step finite even when every
answer is correct.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start at zero. Each step: the fitted probabilities at the current ability, the sum over items
of discrimination times (answer minus probability) minus the ability over the prior variance, the
sum of discrimination squared times p (1 - p) plus one over the prior variance, and move by the
first divided by the second. Stop when the step is tiny, and cap the number of steps.
</details>

In [ ]:
def estimate_theta(y, a, b, prior_sd=1.0, n_iter=50):
    """Posterior-mode ability of one model from its answers `y` on items with fitted `a` and `b`.

    Maximises  sum_i [y_i log p_i + (1 - y_i) log(1 - p_i)]  -  theta^2 / (2 prior_sd^2),
    with p_i = p_correct(theta, a_i, b_i), by Newton's method from theta = 0: at most `n_iter`
    steps, stopping as soon as the step is smaller than 1e-10 in size. `y` holds 0/1 answers (values
    between 0 and 1 are allowed).

    Example:
        >>> abs(estimate_theta([1, 1, 1], [1.0, 1.0, 1.0], [0.0, 0.0, 0.0])) < 3
        True
        (every answer correct: with the prior the ability is large but finite)

    Returns:
        a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement estimate_theta")


def _check_estimate_theta() -> None:
    rng = np.random.default_rng(8)
    a, b = rng.uniform(0.6, 2.5, 40), rng.normal(0.0, 1.0, 40)
    for truth in (-1.5, 0.4, 3.0):
        soft = p_correct(truth, a, b)
        got = estimate_theta(soft, a, b, prior_sd=1000.0)
        assert abs(got - truth) < 1e-4, (
            f"fed the exact probabilities of an ability of {truth} and a very weak prior, the "
            f"estimate should be {truth}, got {got:.4f}. Is the discrimination multiplying each "
            "item's pull, is the prior's pull theta / prior_sd ** 2 (squared), and does the loop "
            "run until the step is tiny, not a few times?")
    far_rng = np.random.default_rng(38)
    far_a, far_b = far_rng.uniform(1.0, 2.0, 60), far_rng.normal(0.0, 1.5, 60)
    far = estimate_theta(p_correct(12.0, far_a, far_b), far_a, far_b, prior_sd=1e8)
    assert abs(far - 12.0) < 1e-6, (
        f"an ability of 12 on items of ordinary difficulty and a very weak prior: expected 12, got {far:.6f}. Newton's "
        "method from zero needs many steps to get that far: stop when the step is tiny, not after a fixed few")
    perfect = estimate_theta(np.ones(40), a, b)
    assert math.isfinite(perfect) and 0 < perfect < 6, (
        f"every answer correct must still give a finite ability (with the prior), got {perfect}")
    y = (rng.random(40) < p_correct(0.7, a, b)).astype(float)
    sd = 0.5
    theta_hat = estimate_theta(y, a, b, prior_sd=sd)
    p = p_correct(theta_hat, a, b)
    slope = float(np.sum(a * (y - p))) - theta_hat / sd ** 2
    assert abs(slope) < 1e-6, (
        f"at the answer the slope of the log-posterior must be zero, yours is {slope:.2e}: the "
        "prior contributes theta / prior_sd^2 to the slope and 1 / prior_sd^2 to the curvature")
    assert abs(estimate_theta(y, a, b, prior_sd=0.2)) < abs(estimate_theta(y, a, b, prior_sd=5.0)), (
        "a tighter prior must pull the estimate closer to zero: is prior_sd used?")
    print("exercise 7 looks right")


_try("exercise 7", _check_estimate_theta, needs=("exercise 1",))

### Exercise 8 — `select_items()`: which hundred?

A hundred items drawn at random are a fair sample. Whether they are the best sample is a
separate question, which the table below measures. Items that discriminate sharply near a model's
ability tell you the most about it, but a model's ability is exactly what you do not know yet.
This lesson's rule trades the two: sort the items by fitted difficulty, cut the sorted list into
`n_select` consecutive blocks, and take the most discriminating item of each block, so the
chosen items cover the whole difficulty range and each is a sharp one.

The blocks are fixed exactly. Sort by `b_hat` ascending, ties by item number. Block `k`
(counting from 0) holds the sorted positions from `floor(k * n / n_select)` up to, not
including, `floor((k + 1) * n / n_select)`, for `n` items. In each block take the item with the
largest `a_hat`; among equal `a_hat` take the one that comes first in the sorted order. Return
the chosen item numbers sorted ascending. Raise `ValueError` if `n_select` is below 1 or above `n`.

<details><summary>💡 Hint 1 — what to think about</summary>

Work in sorted positions first and map back to item numbers last. When `n` is not a multiple of
`n_select` the blocks differ in size by one: the formula for the boundaries says which are larger.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the order that sorts by difficulty, stably, so equal difficulties keep their item order.
For each block, slice that order between the two boundaries, find the largest discrimination
in the slice (the first one wins a tie), and keep that item number. Sort the result at the end.
</details>

In [ ]:
def select_items(a_hat, b_hat, n_select):
    """Choose `n_select` items: the most discriminating item from each of `n_select` consecutive
    blocks of the items sorted by difficulty.

    Sort by `b_hat` ascending, ties by item number. Block k holds sorted positions
    floor(k * n / n_select) to floor((k + 1) * n / n_select) - 1, with n the number of items. Take
    the largest `a_hat` in each block, the first in sorted order on a tie. Raise ValueError if
    `n_select` is below 1 or above n.

    Example:
        >>> select_items([1, 5, 2, 4, 3, 6], [0.5, -1, 2, 0, 1, -2], 3).tolist()
        [3, 4, 5]
        (by difficulty the items run 5, 1, 3, 0, 4, 2; the three blocks are {5, 1}, {3, 0} and
        {4, 2}, whose most discriminating items are 5, 3 and 4)

    Returns:
        an integer array of item numbers, sorted ascending.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement select_items")


def _check_select_items() -> None:
    got = np.asarray(select_items([1, 5, 2, 4, 3, 6], [0.5, -1, 2, 0, 1, -2], 3))
    assert got.tolist() == [3, 4, 5], (
        f"by difficulty the items run 5, 1, 3, 0, 4, 2; the blocks are {{5, 1}}, {{3, 0}}, {{4, 2}} and "
        f"the most discriminating item of each is 5, 3 and 4; got {got.tolist()}")
    a10 = [float(i + 1) for i in range(10)]
    got = np.asarray(select_items(a10, [float(i) for i in range(10)], 4))
    assert got.tolist() == [1, 4, 6, 9], (
        f"ten items in four blocks: the boundaries are floor(k * 10 / 4) = 0, 2, 5, 7, 10, so the blocks hold 2, 3, 2 and 3 "
        f"items and the answers are items 1, 4, 6 and 9; got {got.tolist()}")
    a7 = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
    got = np.asarray(select_items(a7, [float(i) for i in range(7)], 3))
    assert got.tolist() == [1, 3, 6], (
        f"seven items in three blocks: positions 0-1, 2-3 and 4-6 (the boundaries are "
        f"floor(k * 7 / 3) = 0, 2, 4, 7), so the answers are items 1, 3 and 6; got {got.tolist()}")
    got = np.asarray(select_items([2.0, 2.0, 2.0, 2.0], [0.4, 0.1, 0.1, 0.3], 2))
    assert got.tolist() == [1, 3], (
        f"equal discrimination: the first item of each block in sorted order wins (sorted by "
        f"difficulty, ties by item number: 1, 2, 3, 0 -> blocks {{1, 2}} and {{3, 0}}); got {got.tolist()}")
    assert np.asarray(select_items(a7, a7, 7)).tolist() == list(range(7)), "n_select equal to n keeps every item"
    try:
        select_items(a7, a7, 8)
    except ValueError:
        pass
    else:
        raise AssertionError("asking for more items than exist should raise ValueError")
    print("exercise 8 looks right")


_try("exercise 8", _check_select_items)

### Exercise 9 — `p_irt_estimate()`: the answers we have plus the chances we do not

Let `N` be the number of items and `S` the chosen set. The model's realised full score is the sum
of all `N` answers over `N`. On `S` the answers are known. On every other item the best guess of
an answer is its fitted probability at the estimated ability, and the best guess of the sum is the
sum of those probabilities. So

`estimate = (sum of observed answers on S + sum of fitted probabilities on the items not in S) / N`.

Count each item once: an observed item contributes its answer, never its probability as well.
`idx` holds the item numbers of `S` (distinct), and `y_sub[j]` is the answer to item `idx[j]`.

<details><summary>💡 Hint 1 — what to think about</summary>

Two sets of items, observed and not observed, with no overlap and together all `N`. What are
the extreme cases: every item observed, and none observed?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute the fitted probability for all items at the given ability, blank out the observed
ones (mark them with a boolean mask built from `idx`), sum the rest, add the observed answers,
divide by the total number of items.
</details>

In [ ]:
def p_irt_estimate(y_sub, idx, theta, a, b):
    """The p-IRT estimate of a model's full-benchmark accuracy.

    `idx` holds the numbers of the observed items (distinct), `y_sub[j]` is the 0/1 answer to
    item `idx[j]`, `theta` is the model's estimated ability and `a`, `b` are the fitted
    parameters of ALL items. The estimate is the observed answers plus the fitted probabilities
    of the items not observed, divided by the number of items.

    Example:
        >>> p_irt_estimate([1, 0], [0, 2], 0.0, [1.0, 1.0, 1.0, 1.0], [0.0, 0.0, 0.0, 0.0])
        0.5
        (items 1 and 3 each contribute one half; the two observed answers add one: 2 over 4)

    Returns:
        a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement p_irt_estimate")


def _check_p_irt_estimate() -> None:
    a, b = np.ones(4), np.zeros(4)
    got = p_irt_estimate([1, 0], [0, 2], 0.0, a, b)
    assert abs(got - 0.5) < 1e-12, (
        f"two observed answers (one right) and two unseen items at probability one half: "
        f"(1 + 0.5 + 0.5) / 4 = 0.5, got {got}")
    got = p_irt_estimate([1, 1, 0, 1], [0, 1, 2, 3], 2.0, a, b)
    assert abs(got - 0.75) < 1e-12, (
        f"with every item observed the estimate is just the observed accuracy, 0.75, got {got}: an "
        "observed item contributes its answer, not its fitted probability")
    got = p_irt_estimate([], [], 0.0, np.array([1.0, 2.0, 1.0]), np.array([0.0, 0.0, 0.0]))
    assert abs(got - 0.5) < 1e-12, f"with nothing observed it is the mean fitted probability, got {got}"
    rng = np.random.default_rng(9)
    a, b = rng.uniform(0.5, 2.5, 30), rng.normal(0.0, 1.0, 30)
    idx = np.array([21, 3, 17, 8])
    y = np.array([1, 0, 1, 1])
    p = p_correct(0.6, a, b)
    want = (y.sum() + p.sum() - p[idx].sum()) / 30
    got = p_irt_estimate(y, idx, 0.6, a, b)
    assert abs(got - want) < 1e-10, (
        f"expected {want:.6f}, got {got:.6f}: sum the observed answers, add the fitted probabilities "
        "of the items NOT observed only, divide by the number of items of the whole benchmark; "
        "build the unseen mask from idx, which need not be sorted or contiguous")
    print("exercise 9 looks right")


_try("exercise 9", _check_p_irt_estimate, needs=("exercise 1",))

The experiment. For each of the 200 held-out models we know the whole answer sheet, so we know
the target exactly: the share of its 1,000 answers that were right. We pretend to run only the
chosen items, estimate the target two ways (the plain accuracy on the chosen items, and the
p-IRT estimate with your `estimate_theta` and `p_irt_estimate`) and report the root-mean-square
error in accuracy points. Three ways of choosing the items are compared: a random hundred
(twenty of them, averaged), the hundred that carry the most information about an ability of
zero (given helper), and your `select_items`. The last line is a floor: even a method that
knew every item's true parameters and the model's true ability would miss by this much, because
the answers on the unseen items are themselves random.

In [ ]:
def random_items(rng, n_items, k):
    """A random subset of `k` of the `n_items` items, sorted by item number. Given."""
    return np.sort(rng.permutation(n_items)[:k])


def top_information_items(a_hat, b_hat, theta, k):
    """The `k` items with the largest fitted information a^2 p (1 - p) at ability `theta`, ties by
    item number, sorted by item number. Given."""
    p = _world_p(theta, a_hat, b_hat)
    return np.sort(np.argsort(-(a_hat ** 2 * p * (1.0 - p)), kind="stable")[:k])


FULL_TEST = TEST_RESPONSES.mean(axis=1)      # each held-out model's realised full-benchmark accuracy


def _subset_errors(idx):
    """Errors, over the held-out models, of the plain subset accuracy and of the p-IRT estimate
    (YOUR estimate_theta and p_irt_estimate, each call under a time limit). Given."""
    estimate = _guarded(estimate_theta, 2.0, "estimate_theta")
    combine = _guarded(p_irt_estimate, 2.0, "p_irt_estimate")
    plain = TEST_RESPONSES[:, idx].mean(axis=1) - FULL_TEST
    fitted = np.empty(len(TEST_RESPONSES))
    for m in range(len(fitted)):
        y = TEST_RESPONSES[m, idx]
        fitted[m] = combine(y, idx, estimate(y, A_HAT[idx], B_HAT[idx]), A_HAT, B_HAT)
    return plain, fitted - FULL_TEST


def _rms(err):
    return float(np.sqrt(np.mean(np.asarray(err) ** 2)))


def show_cheap_estimates() -> None:
    thirds = np.array_split(np.argsort(TEST_THETA, kind="stable"), 3)
    rng = np.random.default_rng([SEED, 30])
    rows = []
    plain_all, fitted_all = [], []
    for _ in range(20):
        plain, fitted = _subset_errors(random_items(rng, N_ITEMS, K_SUBSET))
        plain_all.append(plain)
        fitted_all.append(fitted)
    rows.append((f"random {K_SUBSET} (mean over 20 draws)",
                 float(np.mean([_rms(e) for e in plain_all])), float(np.mean([_rms(e) for e in fitted_all])),
                 [float(np.mean([_rms(e[t]) for e in fitted_all])) for t in thirds]))
    for name, idx in ((f"the {K_SUBSET} most informative at ability 0",
                       top_information_items(A_HAT, B_HAT, 0.0, K_SUBSET)),
                      (f"{K_SUBSET} stratified by difficulty (exercise 8)",
                       select_items(A_HAT, B_HAT, K_SUBSET))):
        plain, fitted = _subset_errors(idx)
        rows.append((name, _rms(plain), _rms(fitted), [_rms(fitted[t]) for t in thirds]))
    print(f"{len(TEST_RESPONSES)} held-out models; root-mean-square error of the estimated full "
          f"accuracy, in points (1 point = 1 percentage point of the {N_ITEMS} items)")
    print(f"  {'items run':<44} {'plain accuracy':>15} {'p-IRT':>7}   p-IRT by ability: weakest, middle, strongest third")
    for name, plain, fitted, by_third in rows:
        print(f"  {name:<44} {100 * plain:>15.2f} {100 * fitted:>7.2f}   "
              + "  ".join(f"{100 * v:>6.2f}" for v in by_third))
    idx = select_items(A_HAT, B_HAT, K_SUBSET)
    unseen = np.ones(N_ITEMS, dtype=bool)
    unseen[idx] = False
    p_true = _world_p(TEST_THETA[:, None], BANK_A[None, :], BANK_B[None, :])
    floor = _rms((TEST_RESPONSES[:, unseen] - p_true[:, unseen]).sum(axis=1) / N_ITEMS)
    print(f"  {'floor: true parameters, the stratified items':<44} {'':>15} {100 * floor:>7.2f}")


_demo("cheap estimates", show_cheap_estimates,
      needs=("exercise 1", "exercise 7", "exercise 8", "exercise 9"))

Read the table with these questions. Which choice of items has the largest error in plain
accuracy, and what does p-IRT do for it? Why can the plain accuracy on a hand-picked subset be
wrong in a way a random subset's is not? In which third of the ability range does each p-IRT
estimate do worst, and does that match what you chose the items for? Last, how much of the best
row's error is the floor, which no choice of items can remove?

One more question, this time about the twelve models of the leaderboard: how many of their
pairs does a hundred-item estimate put in the same order as the full benchmark?

In [ ]:
def show_leaderboard_from_100() -> None:
    full = RESPONSES[0].sum(axis=1)
    pairs = [(i, j) for i in range(N_MODELS) for j in range(i + 1, N_MODELS) if full[i] != full[j]]
    estimate = _guarded(estimate_theta, 2.0, "estimate_theta")
    combine = _guarded(p_irt_estimate, 2.0, "p_irt_estimate")

    def judged(est):
        """(pairs ordered as the full benchmark orders them, widest full-benchmark gap among the
        pairs that were not), a tie in the estimate counting as not ordered."""
        wrong = [(i, j) for i, j in pairs if np.sign(est[i] - est[j]) != np.sign(full[i] - full[j])]
        return len(pairs) - len(wrong), max((abs(full[i] - full[j]) / N_ITEMS for i, j in wrong), default=0.0)

    def p_irt_scores(idx):
        out = []
        for m in range(N_MODELS):
            y = RESPONSES[0, m, idx]
            out.append(combine(y, idx, estimate(y, A_HAT[idx], B_HAT[idx]), A_HAT, B_HAT))
        return np.array(out)

    rng = np.random.default_rng([SEED, 31])
    plain, fitted = [], []
    for _ in range(20):
        idx = random_items(rng, N_ITEMS, K_SUBSET)
        plain.append(judged(RESPONSES[0][:, idx].mean(axis=1)))
        fitted.append(judged(p_irt_scores(idx)))

    def over_draws(draws):
        """The mean over the draws of each column, with its lowest and highest draw."""
        ok, gap = [r[0] for r in draws], [r[1] for r in draws]
        return (f"{np.mean(ok):.1f} ({min(ok)} to {max(ok)})",
                f"{100 * np.mean(gap):.1f} ({100 * min(gap):.1f} to {100 * max(gap):.1f})")

    ok, gap = judged(p_irt_scores(select_items(A_HAT, B_HAT, K_SUBSET)))
    rows = [(f"random {K_SUBSET}, plain accuracy: mean of 20 draws", *over_draws(plain)),
            (f"random {K_SUBSET}, p-IRT: mean of 20 draws", *over_draws(fitted)),
            (f"stratified {K_SUBSET} (exercise 8), p-IRT: ONE subset", str(ok), f"{100 * gap:.1f}")]
    print(f"the {N_MODELS} leaderboard models in format 0: {len(pairs)} pairs with different full-benchmark scores; "
          "for the random rows, the lowest and highest of the 20 draws are in brackets")
    print(f"  {'estimate from the chosen items':<52} {'pairs in the full order':>24} {'widest misordered gap, pts':>30}")
    for name, n_ok, widest in rows:
        print(f"  {name:<52} {n_ok:>24} {widest:>30}")


_demo("leaderboard from 100", show_leaderboard_from_100,
      needs=("exercise 1", "exercise 7", "exercise 8", "exercise 9"))

Read the three rows before deciding anything. The two random rows are means over twenty different
subsets, with the lowest and highest draw beside them; the stratified row is ONE subset, a single
draw, so set it against that range and not against the mean alone. Does the better estimate of
each model's score (the p-IRT rows of the table above) put close pairs in the right order more
often? How wide is the widest gap among the misordered pairs, beside the smallest gap that
section 5 said the full benchmark can detect near the middle of the scale?

## 10. Common mistakes

* **Trusting the sample-size formula as a guarantee.** It assumes the per-item variance and the
  gap are known. Confirm a plan by simulation (section 6) before you spend the budget on it.
* **Sizing a paired comparison with the unpaired variance, or the reverse.** The two differ by the
  covariance of the models' per-item chances (section 4), which can change the answer by a lot.
* **Counting a replicate at exactly zero as support for a gap.** The interval must exclude zero.
* **Resampling the items separately for each model.** A leaderboard scores everyone on the same
  items, so a bootstrap must draw ONE set of items per replicate. The cell below shows what
  the other choice does to a gap.
* **Reporting a rank without its range, or a range from one source of doubt.** Items and prompt
  format move a model by different amounts.
* **Averaging fitted probabilities over all items and ignoring the answers you have.** The target
  is the score the model really got.
* **Reading the plain accuracy of a hand-picked subset as a sample.** It is a sample only if the
  items were drawn to represent the rest.
* **Fitting item parameters on the models you then evaluate.** The held-out models here were
  never in the fit; a table computed on models that were in it would test nothing.

In [ ]:
def show_unpaired_trap() -> None:
    a_model, b_model = MODEL_NAMES.index("Zephyr"), MODEL_NAMES.index("Wren")
    x_a, x_b = RESPONSES[0, a_model], RESPONSES[0, b_model]
    rng = np.random.default_rng([SEED, 40])
    paired = paired_bootstrap_sums(x_a, x_b, N_BOOT, rng)
    grid_a = rng.integers(0, N_ITEMS, size=(N_BOOT, N_ITEMS))
    grid_b = rng.integers(0, N_ITEMS, size=(N_BOOT, N_ITEMS))
    unpaired = x_a[grid_a].sum(axis=1) - x_b[grid_b].sum(axis=1)
    observed = int(x_a.sum() - x_b.sum())
    print(f"{MODEL_NAMES[a_model]} against {MODEL_NAMES[b_model]}, format 0, {N_ITEMS} items; "
          f"observed gap {100 * observed / N_ITEMS:+.1f} points; {N_BOOT} replicates")
    print(f"  {'bootstrap':<34} {'SD of the gap':>14} {'95% interval':>22} {'replicates at or below 0':>26}")
    for name, sums in (("one item draw for both (paired)", paired), ("separate draws per model", unpaired)):
        lo, hi = np.percentile(sums, [2.5, 97.5]) / N_ITEMS * 100
        print(f"  {name:<34} {100 * np.std(sums) / N_ITEMS:>11.2f} pts {f'[{lo:+.1f}, {hi:+.1f}] pts':>22} "
              f"{int(np.sum(sums <= 0)):>26}")


_demo("unpaired trap", show_unpaired_trap)

## 11. Self-check

1. You score two models on the same questions and compare the measured gap with the gap you
   would get scoring them on separate random questions. The shared design has the smaller
   variance when:
   - (a) the two models have similar overall accuracy on the benchmark as a whole
   - (b) the questions that are hard for one model tend to be hard for the other
   - (c) the questions are shuffled into a different order for each of the two models
   - (d) the benchmark has more than a thousand questions in total

2. A plan says a gap of 4 points needs a certain number of items for 80% power. You now want
   to detect a gap of 2 points at the same level and power, with the same per-item variance.
   You should plan for about:
   - (a) half as many items
   - (b) the same number of items
   - (c) four times as many items
   - (d) twice as many items

3. A leaderboard puts model X at rank 5. Resampling the items moves X between ranks 5 and 6;
   resampling the prompt format moves it between ranks 3 and 9. The best-supported reading is:
   - (a) rank 5 is reliable, because resampling the items moves it by only one place at most
   - (b) the format range is irrelevant, because a benchmark is run with one fixed prompt format
   - (c) the item range is the whole uncertainty, because a format shifts every model by the same amount
   - (d) the item range is one source of doubt; a rank claim should also say how far the format moves it

4. You run 100 hand-picked items and report the plain accuracy on them as an estimate of the
   full-benchmark score. Why can that be wrong even when the 100 items are good ones?
   - (a) They were chosen for what they reveal, not to represent the rest, so the share right on them is biased
   - (b) A hundred items can never be enough, because accuracy on a hand-picked subset cannot be corrected
   - (c) Plain accuracy is rounded to the nearest percentage point, which hurts hand-picked items most
   - (d) A model that answers a hand-picked item correctly has been lucky, so every model is overstated

5. The p-IRT estimator adds the observed answers to the fitted probabilities of the unseen items,
   instead of averaging the fitted probabilities over all items. Why?
   - (a) To make the estimate larger, because adding observed answers can only raise it above the average
   - (b) The target is the score the model actually got: the observed part is known, only the rest is predicted
   - (c) Because fitted probabilities are unreliable, so as few of them as possible should enter the estimate
   - (d) Because the unseen items are always the easier ones, so their fitted probabilities would be too low

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)"
                if status == "not started" and waiting not in (None, "", function) else "")
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]} — `{next_up[1]}`, {next_up[2]}. Its cell has hints if you "
              "are stuck.")


def _finish() -> None:
    """Re-run every exercise check quietly, so the board is current even if you edited an
    exercise and did not re-run its cell; then the board, then the verdict. A stub you have not
    reached yet is not a failure; a check that ran and came back wrong is. In a script or under CI
    that ends the run non-zero, so a green exit code can never paper over it. In a notebook kernel
    it is a printed line."""
    checks = (("exercise 1", _check_p_correct, ()),
              ("exercise 2", _check_diff_variances, ()),
              ("exercise 3", _check_items_needed, ()),
              ("exercise 4", _check_bootstrap_detects, ()),
              ("exercise 5", _check_resample_scores, ()),
              ("exercise 6", _check_rank_counts, ()),
              ("exercise 7", _check_estimate_theta, ("exercise 1",)),
              ("exercise 8", _check_select_items, ()),
              ("exercise 9", _check_p_irt_estimate, ("exercise 1",)))
    with contextlib.redirect_stdout(io.StringIO()):
        for label, check, needs in checks:
            _try(label, check, needs)
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed" and label in _FUNCTION]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake "
              "where it ran.")


if __name__ == "__main__":
    _finish()

## What you built, and where it goes next

You wrote the model of a benchmark, the variance of a measured gap with and without sharing
items, the number of items a stated gap needs, a paired-bootstrap verdict, a bootstrap over items
and prompt formats with a rank table, and a three-part cheap estimate of a full score. Together
they let you say, for a leaderboard, how small a gap its size can resolve, how far each rank can
move, and what a hundred items can and cannot tell you.

What this lesson left out on purpose: models that agree beyond what item difficulty explains,
more than one ability dimension, clustered questions (items that share a passage), adaptive
stopping rules that decide the number of items as the data arrive, new models whose abilities
lie outside those of the reference models (here both come from the same distribution), and
real model outputs. As of 14 October 2026 the tinyBenchmarks paper and its released tools
(`claims.yaml`) are where the production versions of the estimators live. `T10-L03` asks how
far a ranking moves for the other common kind of leaderboard, one built from pairwise votes.
The habit to take from here is to run a method on a world where you know the answer before you
trust it on one where you do not.

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.